In [1]:
# Cell 1: Environment setup (GPU-safe)
import os
import random
import importlib
import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    set_seed,
    default_data_collator,
    logging as hf_logging,
 )

hf_logging.set_verbosity_error()

device = "cuda" if torch.cuda.is_available() else "cpu"
use_fp16 = torch.cuda.is_available()
seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
set_seed(seed)

print(f"Device: {device}")
print("GPU:", torch.cuda.get_device_name(0) if device == "cuda" else "CPU")
print(f"FP16 enabled: {use_fp16}")
print("Seeds initialized")

Device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
FP16 enabled: True
Seeds initialized


In [2]:
# Cell 2: Paths + dataset existence check
ARTIFACT_DIR = "./artifacts"
DISTIL_LOCAL_PATH = "./models/distilgpt2"
TINY_LOCAL_PATH = "./models/tinyllama"
DATASET_PATH = os.path.join(ARTIFACT_DIR, "teacher_student_genai_dataset.csv")

DISTIL_OUTPUT_DIR = os.path.join(ARTIFACT_DIR, "distilgpt2_finetuned")
TINY_LORA_OUTPUT_DIR = os.path.join(ARTIFACT_DIR, "tinyllama_lora_adapter")

os.makedirs(ARTIFACT_DIR, exist_ok=True)

if not os.path.exists(DATASET_PATH):
    print(f"Dataset not found at {DATASET_PATH}")
    print("Creating a fallback synthetic dataset...")
    fallback_data = [
        ("Student: What is a noun?", "Teacher: A noun is a word that names a person place or thing."),
        ("Student: What is a verb?", "Teacher: A verb describes an action or state."),
        ("Student: What is an adjective?", "Teacher: An adjective describes a noun."),
        ("Student: What is gravity?", "Teacher: Gravity is the force that attracts objects with mass."),
        ("Student: What is a planet?", "Teacher: A planet is a large celestial body that orbits a star."),
        ("Student: What is photosynthesis?", "Teacher: Photosynthesis is the process plants use to convert sunlight into energy."),
        ("Student: What is addition?", "Teacher: Addition is combining numbers to get a total."),
        ("Student: What is subtraction?", "Teacher: Subtraction finds the difference between numbers."),
    ]
    fallback_df = pd.DataFrame([{"text": q + "\n" + a} for q, a in fallback_data])
    fallback_df.to_csv(DATASET_PATH, index=False)
    print(f"Fallback dataset saved to {DATASET_PATH}")
else:
    print(f"Dataset found at {DATASET_PATH}")

if not os.path.exists(DISTIL_LOCAL_PATH):
    raise FileNotFoundError(f"Local DistilGPT2 path not found: {DISTIL_LOCAL_PATH}")
if not os.path.exists(TINY_LOCAL_PATH):
    raise FileNotFoundError(f"Local TinyLlama path not found: {TINY_LOCAL_PATH}")

print(f"Local DistilGPT2 path: {DISTIL_LOCAL_PATH}")
print(f"Local TinyLlama path: {TINY_LOCAL_PATH}")

Dataset found at ./artifacts\teacher_student_genai_dataset.csv
Local DistilGPT2 path: ./models/distilgpt2
Local TinyLlama path: ./models/tinyllama


In [3]:
# Cell 3: Load dataset using pandas (no datasets library)
df = pd.read_csv(DATASET_PATH)

if "text" not in df.columns:
    raise ValueError("Dataset must contain a 'text' column")

texts = df["text"].dropna().astype(str).tolist()

if len(texts) == 0:
    raise ValueError("No training samples found in dataset")

print(f"Loaded {len(texts)} samples from {DATASET_PATH}")
df.head()

Loaded 8 samples from ./artifacts\teacher_student_genai_dataset.csv


,text
0,Student: What is a noun?\nTeacher: A noun is a...
1,Student: What is a verb?\nTeacher: A verb desc...
2,Student: What is an adjective?\nTeacher: An ad...
3,Student: What is gravity?\nTeacher: Gravity is...
4,Student: What is a planet?\nTeacher: A planet ...


In [4]:
# Cell 4: Model registry (local paths only)
models = {
    "distilgpt2": DISTIL_LOCAL_PATH,
    "tinyllama": TINY_LOCAL_PATH,
}
models

{'distilgpt2': './models/distilgpt2', 'tinyllama': './models/tinyllama'}

In [5]:
# Cell 5: Tokenizer loader (pad/eos handling)
def load_tokenizer(local_model_path):
    tokenizer = AutoTokenizer.from_pretrained(local_model_path, local_files_only=True, use_fast=True)

    if tokenizer.eos_token is None:
        tokenizer.add_special_tokens({"eos_token": "<|eos|>"})

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "right"
    return tokenizer

In [6]:
# Cell 6: Tokenization + torch dataset
MAX_LENGTH = 128

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __len__(self):
        return self.encodings["input_ids"].shape[0]

    def __getitem__(self, idx):
        item = {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
        }
        item["labels"] = item["input_ids"].clone()
        return item

def build_text_dataset(text_list, tokenizer, max_length=MAX_LENGTH):
    encodings = tokenizer(
        text_list,
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_tensors="pt",
    )
    return TextDataset(encodings)

In [ ]:
# Cell 7: DistilGPT2 local load + full fine-tuning
# Cell 7: Shared utilities (updated)

# Cell 7: Shared utilities (FINAL VERSION with gradient_accumulation_steps)

from torch.utils.data import Dataset
from transformers import DataCollatorForLanguageModeling

class TextDataset(Dataset):
    def __init__(self, tokenized_texts):
        self.input_ids = tokenized_texts["input_ids"]
        self.attention_mask = tokenized_texts["attention_mask"]

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.input_ids[idx],
        }


def build_text_dataset(text_list, tokenizer, max_length=256):
    encodings = tokenizer(
        text_list,
        truncation=True,
        padding=True,
        max_length=max_length,
        return_tensors="pt",
    )
    return TextDataset(encodings)


def make_training_args(
    output_dir,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    learning_rate=5e-5,
    logging_steps=10,
    fp16=False,
    gradient_accumulation_steps=1,  # <-- ADDED THIS LINE
):
    common_kwargs = {
        "output_dir": output_dir,
        "num_train_epochs": num_train_epochs,
        "per_device_train_batch_size": per_device_train_batch_size,
        "learning_rate": learning_rate,
        "logging_steps": logging_steps,
        "fp16": fp16,
        "gradient_accumulation_steps": gradient_accumulation_steps,  # <-- ADDED THIS LINE
    }

    return TrainingArguments(
        save_strategy="epoch",
        report_to="none",
        remove_unused_columns=False,
        **common_kwargs,
    )


def load_local_causal_lm(model_path, tokenizer, fp16=True):
    print(f"Fine-tuning {model_path.split('/')[-1]} from local folder...")
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.float16 if fp16 and torch.cuda.is_available() else torch.float32,
    )
    model.resize_token_embeddings(len(tokenizer))
    model.to(device)
    return model


In [8]:
# Cell 8: TinyLlama local load + LoRA fine-tuning only
def attach_lora_to_tinyllama(base_model):
    try:
        peft = importlib.import_module("peft")
    except ModuleNotFoundError as err:
        raise ModuleNotFoundError(
            "peft is required for LoRA fine-tuning. Install it with: pip install peft"
        ) from err

    lora_cfg = peft.LoraConfig(
        task_type=peft.TaskType.CAUSAL_LM,
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )
    lora_model = peft.get_peft_model(base_model, lora_cfg)
    lora_model.print_trainable_parameters()
    return lora_model

def fine_tune_tinyllama_lora(text_list, num_train_epochs=1):
    print("\nApplying LoRA fine-tuning on TinyLlama from local folder...")
    tokenizer = load_tokenizer(TINY_LOCAL_PATH)
    base_model = load_local_causal_lm(TINY_LOCAL_PATH, tokenizer, fp16=use_fp16)
    lora_model = attach_lora_to_tinyllama(base_model)
    dataset = build_text_dataset(text_list, tokenizer)

    training_args = make_compatible_training_args(
        output_dir=TINY_LORA_OUTPUT_DIR,
        num_train_epochs=num_train_epochs,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        fp16=use_fp16,
    )

    trainer = Trainer(
        model=lora_model,
        args=training_args,
        train_dataset=dataset,
        data_collator=default_data_collator,
    )

    trainer.train()
    lora_model.save_pretrained(TINY_LORA_OUTPUT_DIR)
    tokenizer.save_pretrained(TINY_LORA_OUTPUT_DIR)
    print(f"Saved TinyLlama LoRA adapter to: {TINY_LORA_OUTPUT_DIR}")

    return lora_model, tokenizer

In [9]:
# Cell 9: Run DistilGPT2 full fine-tuning
# Cell 9: Run DistilGPT2 full fine-tuning (updated)

# Cell 9: Run DistilGPT2 full fine-tuning (updated for older Trainer API)

# Cell 9: Run DistilGPT2 full fine-tuning (updated for older Trainer API)

# Cell 9: Run DistilGPT2 full fine-tuning (FP16 disabled for stability)

from transformers import Trainer

def fine_tune_distilgpt2(
    text_list,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    learning_rate=5e-5,
):
    # Load tokenizer from local DistilGPT2 path
    tokenizer = AutoTokenizer.from_pretrained(DISTIL_LOCAL_PATH)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Build dataset
    dataset = build_text_dataset(text_list, tokenizer)

    # Load model in FULL FP32 (no FP16) to avoid gradient scaling issues
    model = AutoModelForCausalLM.from_pretrained(
        DISTIL_LOCAL_PATH,
        torch_dtype=torch.float32,  # Force FP32
    )
    model.resize_token_embeddings(len(tokenizer))
    model.to(device)

    # Training arguments: FP16=OFF, full precision
    training_args = make_training_args(
        output_dir=DISTIL_OUTPUT_DIR,
        num_train_epochs=num_train_epochs,
        per_device_train_batch_size=per_device_train_batch_size,
        learning_rate=learning_rate,
        logging_steps=1,
        fp16=False,  # DISABLED
    )

    # Data collator for causal LM
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
        data_collator=data_collator,
    )

    trainer.train()
    trainer.save_model(DISTIL_OUTPUT_DIR)
    tokenizer.save_pretrained(DISTIL_OUTPUT_DIR)

    print(f"DistilGPT2 fine-tuned model saved to {DISTIL_OUTPUT_DIR}")
    return model, tokenizer


# Actually run fine-tuning
distil_model, distil_tokenizer = fine_tune_distilgpt2(texts, num_train_epochs=2)


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

{'loss': '4.047', 'grad_norm': '25.1', 'learning_rate': '5e-05', 'epoch': '0.25'}
{'loss': '3.343', 'grad_norm': '18.67', 'learning_rate': '4.375e-05', 'epoch': '0.5'}
{'loss': '2.848', 'grad_norm': '22.67', 'learning_rate': '3.75e-05', 'epoch': '0.75'}
{'loss': '2.48', 'grad_norm': '16.91', 'learning_rate': '3.125e-05', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '2.274', 'grad_norm': '16.25', 'learning_rate': '2.5e-05', 'epoch': '1.25'}
{'loss': '2.63', 'grad_norm': '16.54', 'learning_rate': '1.875e-05', 'epoch': '1.5'}
{'loss': '2.146', 'grad_norm': '17.02', 'learning_rate': '1.25e-05', 'epoch': '1.75'}
{'loss': '2.43', 'grad_norm': '17.52', 'learning_rate': '6.25e-06', 'epoch': '2'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '2.852', 'train_samples_per_second': '5.611', 'train_steps_per_second': '2.805', 'train_loss': '2.775', 'epoch': '2'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

DistilGPT2 fine-tuned model saved to ./artifacts\distilgpt2_finetuned


In [12]:
# Cell 10: Run TinyLlama LoRA fine-tuning (fixed)

# First import peft if not already imported
try:
    from peft import LoraConfig, get_peft_model, TaskType
    print("PEFT imported successfully")
except ImportError:
    print("Installing peft...")
    import sys
    !{sys.executable} -m pip install peft
    from peft import LoraConfig, get_peft_model, TaskType

def attach_lora_to_tinyllama(base_model):
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        inference_mode=False,
        r=16,
        lora_alpha=32,
        lora_dropout=0.1,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    )
    lora_model = get_peft_model(base_model, lora_config)
    lora_model.print_trainable_parameters()
    return lora_model

def fine_tune_tinyllama_lora(
    text_list,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    learning_rate=1e-4,
):
    print("Applying LoRA fine-tuning on TinyLlama from local folder...")
    
    # Load tokenizer and base model
    tokenizer = AutoTokenizer.from_pretrained(TINY_LOCAL_PATH)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    base_model = AutoModelForCausalLM.from_pretrained(
        TINY_LOCAL_PATH,
        torch_dtype=torch.float32,  # Full precision like DistilGPT2
    )
    base_model.resize_token_embeddings(len(tokenizer))
    base_model.to(device)
    
    # Apply LoRA
    lora_model = attach_lora_to_tinyllama(base_model)
    
    # Build dataset
    dataset = build_text_dataset(text_list, tokenizer)
    
    # Use the SAME make_training_args we fixed earlier (with fp16=False)
    training_args = make_training_args(
        output_dir=TINY_LORA_OUTPUT_DIR,
        num_train_epochs=num_train_epochs,
        per_device_train_batch_size=per_device_train_batch_size,
        learning_rate=learning_rate,
        logging_steps=1,
        fp16=False,  # Full precision to avoid gradient issues
        gradient_accumulation_steps=4,
    )
    
    # Use default collator (like original code intended)
    from transformers import default_data_collator
    
    trainer = Trainer(
        model=lora_model,
        args=training_args,
        train_dataset=dataset,
        data_collator=default_data_collator,
    )
    
    trainer.train()
    trainer.save_model(TINY_LORA_OUTPUT_DIR)
    tokenizer.save_pretrained(TINY_LORA_OUTPUT_DIR)
    
    print(f"TinyLlama LoRA adapter saved to {TINY_LORA_OUTPUT_DIR}")
    return lora_model, tokenizer


# Actually run LoRA fine-tuning
tiny_lora_model, tiny_tokenizer = fine_tune_tinyllama_lora(texts, num_train_epochs=1)


PEFT imported successfully
Applying LoRA fine-tuning on TinyLlama from local folder...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


TypeError: make_training_args() got an unexpected keyword argument 'gradient_accumulation_steps'

In [ ]:
# Cell 11: Safe text generation helper
def safe_generate_text(
    model,
    tokenizer,
    prompt,
    max_new_tokens=60,
    temperature=0.7,
    top_p=0.9,
    max_input_length=128,
 ):
    if not isinstance(prompt, str) or not prompt.strip():
        return "Invalid prompt: please provide a non-empty string."

    model.eval()
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_length,
        padding=True,
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    try:
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temperature,
                top_p=top_p,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        return tokenizer.decode(output[0], skip_special_tokens=True)
    except RuntimeError as err:
        if "out of memory" in str(err).lower() and torch.cuda.is_available():
            torch.cuda.empty_cache()
            return "Generation failed due to CUDA OOM. Try smaller max_new_tokens."
        raise

In [ ]:
# Cell 12: Example prompts for both models
test_prompts = [
    "Student: What is a noun?\nTeacher:",
    "Student: Explain gravity in simple words.\nTeacher:",
    "Student: What is photosynthesis?\nTeacher:",
    "Student: Give me one math addition example.\nTeacher:",
]
test_prompts

In [ ]:
# Cell 13: Run generation tests for DistilGPT2 and TinyLlama LoRA
print("DISTILGPT2 (Fine-tuned)\n")
for prompt in test_prompts:
    print("Prompt:", prompt)
    print(safe_generate_text(distil_model, distil_tokenizer, prompt))
    print("-" * 70)

print("\nTINYLLAMA (LoRA Adapter)\n")
for prompt in test_prompts:
    print("Prompt:", prompt)
    print(safe_generate_text(tiny_lora_model, tiny_tokenizer, prompt))
    print("-" * 70)

In [ ]:
# Cell 14: Simple evaluation + comparison (DistilGPT2 vs TinyLlama LoRA)
from difflib import SequenceMatcher

eval_samples = [
    {
        "prompt": "Student: What is a noun?\nTeacher:",
        "reference": "A noun is a word that names a person place or thing."
    },
    {
        "prompt": "Student: Explain gravity in simple words.\nTeacher:",
        "reference": "Gravity is the force that pulls objects toward each other."
    },
    {
        "prompt": "Student: What is photosynthesis?\nTeacher:",
        "reference": "Photosynthesis is how plants use sunlight to make food."
    },
    {
        "prompt": "Student: Give me one math addition example.\nTeacher:",
        "reference": "An example is 2 + 3 = 5."
    },
]

def _normalize_text(s):
    return " ".join(str(s).strip().lower().split())

def simple_score(generated, reference):
    gen = _normalize_text(generated)
    ref = _normalize_text(reference)

    # Token overlap proxy
    gen_tokens = set(gen.split())
    ref_tokens = set(ref.split())
    overlap = len(gen_tokens & ref_tokens) / max(len(ref_tokens), 1)

    # Character-level similarity proxy
    seq_sim = SequenceMatcher(None, gen, ref).ratio()

    # Combined score (0 to 1)
    score = 0.6 * overlap + 0.4 * seq_sim
    return round(score, 4), round(overlap, 4), round(seq_sim, 4)

rows = []
for sample in eval_samples:
    prompt = sample["prompt"]
    reference = sample["reference"]

    distil_out = safe_generate_text(distil_model, distil_tokenizer, prompt, max_new_tokens=60)
    tiny_out = safe_generate_text(tiny_lora_model, tiny_tokenizer, prompt, max_new_tokens=60)

    d_score, d_overlap, d_seq = simple_score(distil_out, reference)
    t_score, t_overlap, t_seq = simple_score(tiny_out, reference)

    rows.append({
        "prompt": prompt.replace("\n", " "),
        "reference": reference,
        "distil_score": d_score,
        "tinyllama_lora_score": t_score,
        "winner": "distilgpt2" if d_score > t_score else ("tinyllama_lora" if t_score > d_score else "tie"),
    })

eval_df = pd.DataFrame(rows)
print("Average DistilGPT2 score:", round(eval_df["distil_score"].mean(), 4))
print("Average TinyLlama LoRA score:", round(eval_df["tinyllama_lora_score"].mean(), 4))
eval_df

In [ ]:
# Cell 15: Environment dependency quick-check (run once in your conda env)
required_packages = [
    "torch",
    "transformers",
    "accelerate",
    "peft",
    "pandas",
    "numpy",
    "sentencepiece",
    "safetensors",
]

optional_packages = [
    "bitsandbytes",
    "xformers",
]

def check_packages(packages):
    status = []
    for pkg in packages:
        try:
            importlib.import_module(pkg)
            status.append((pkg, "OK"))
        except Exception:
            status.append((pkg, "MISSING"))
    return pd.DataFrame(status, columns=["package", "status"] )

print("Required package status:")
required_df = check_packages(required_packages)
display(required_df)

missing_required = required_df[required_df["status"] == "MISSING"]["package"].tolist()
if missing_required:
    print("\nInstall missing required packages with:")
    print("pip install " + " ".join(missing_required))
else:
    print("\nAll required packages are installed.")

print("\nOptional package status:")
display(check_packages(optional_packages))